In [ ]:
!git config --global user.email 'giudittamacigno03@gmail.com'
!git config --global user.name 'flaviofrasca'

import os

!git clone https://github.com/flaviofrasca/MaskArchitectureAnomaly_CourseProject.git
os.chdir('/content/MaskArchitectureAnomaly_CourseProject/eomt')
print(os.getcwd())

from google.colab import drive
drive.mount('/content/drive')

!pip install -q \
    "lightning==2.5.1.post0" \
    "timm==1.0.15" \
    "transformers==4.56.1" \
    "torchmetrics==1.7.1" \
    "jsonargparse[signatures]==4.38" \
    "pycocotools==2.0.8" \
    "fvcore==0.1.5.post20221221" \
    "wandb==0.19.10" \
    "scipy==1.15.2" \
    "gitignore_parser==0.1.12"

Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 266 (delta 32), reused 25 (delta 25), pack-reused 214 (from 4)
Receiving objects: 100% (266/266), 27.72 MiB | 18.64 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/MaskArchitectureAnomaly_CourseProject/eomt
Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.

In [ ]:
%%javascript
function keepAlive() {
    console.log("Keep-alive: " + new Date().toLocaleTimeString());
    window.scrollBy(0, 1);
    window.scrollBy(0, -1);
    setTimeout(keepAlive, 60000);
}
keepAlive();

<IPython.core.display.Javascript object>

In [ ]:
import os, glob, torch
import torch.nn.functional as F
os.environ["WANDB_MODE"] = "disabled"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
COCO_CKPT = BASE + "/checkpoints/coco/eomt_coco.bin"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

def find_last_checkpoint(phase_dir):
    ckpts = glob.glob(f"{phase_dir}/*/.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint in {phase_dir}")
    return sorted(ckpts, key=os.path.getmtime)[-1]

In [ ]:
import os, glob, torch
import torch.nn.functional as F
os.environ["WANDB_MODE"] = "disabled"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
COCO_CKPT = BASE + "/checkpoints/coco/eomt_coco.bin"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

def find_last_checkpoint(phase_dir):

    ckpts = glob.glob(f"{phase_dir}/*/.ckpt", recursive=True)


    if not ckpts:
        ckpts = glob.glob(f"{phase_dir}/*.ckpt")

    if not ckpts:
        raise FileNotFoundError(f"🚨 Nessun checkpoint trovato in {phase_dir}")

    return sorted(ckpts, key=os.path.getmtime)[-1]

In [ ]:
import torch
import torch.nn.functional as F

ckpt = torch.load(COCO_CKPT, map_location="cpu", weights_only=False)
state_dict = ckpt.get("state_dict", ckpt)


state_dict.pop("criterion.empty_weight", None)


pos_embed = state_dict["network.encoder.backbone.pos_embed"]
h_old, w_old, h_new, w_new = 40, 40, 64, 64
pos_embed_4d = pos_embed.reshape(1, h_old, w_old, 768).permute(0, 3, 1, 2).float()
pos_embed_interp = F.interpolate(pos_embed_4d, size=(h_new, w_new), mode='bicubic', align_corners=False)
state_dict["network.encoder.backbone.pos_embed"] = pos_embed_interp.permute(0, 2, 3, 1).reshape(1, h_new * w_new, 768)
print(f"pos_embed: {pos_embed.shape} → {state_dict['network.encoder.backbone.pos_embed'].shape}")


q_weight = state_dict["network.q.weight"]
state_dict["network.q.weight"] = q_weight[:100, :]
print(f"q.weight: {q_weight.shape} → {state_dict['network.q.weight'].shape}")

FILTERED_COCO_CKPT = "/tmp/coco_interpolated.bin"
torch.save(state_dict, FILTERED_COCO_CKPT)
print(f"Checkpoint pronto → {FILTERED_COCO_CKPT}")

pos_embed: torch.Size([1, 1600, 768]) → torch.Size([1, 4096, 768])
q.weight: torch.Size([200, 768]) → torch.Size([100, 768])
Checkpoint pronto → /tmp/coco_interpolated.bin


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

phase_dir = f"{SAVE_DIR}/phase1_head_only"
os.makedirs(phase_dir, exist_ok=True)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {FILTERED_COCO_CKPT}"
    f" --model.init_args.load_ckpt_class_head False"
    f" --model.init_args.llrd 0.0"
    f" --model.init_args.lr_mult 0.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 3"
    f" --trainer.default_root_dir {phase_dir}"
    ' "--trainer.callbacks+={\"class_path\": \"lightning.pytorch.callbacks.ModelCheckpoint\", \"init_args\": {\"save_last\": true, \"every_n_epochs\": 1}}"'
    f" --compile_disabled"
)
!{cmd}


2026-05-25 14:58:29.958459: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 195 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

   | Name                     | Type                        | Params | Mode 
----------------------------------------------------------------------------------
0  | netwo

FileNotFoundError: Nessun checkpoint in /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase1_head_only"
os.makedirs(phase_dir, exist_ok=True)

CHUNK1_CKPT = f"{phase_dir}/epoch=2-step=2232.ckpt"

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {CHUNK1_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.0"
    f" --model.init_args.lr_mult 0.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 3"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}


2026-05-26 14:02:25.679867: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treesp

In [ ]:
import os, glob, re, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase1_head_only"

def find_best_epoch_checkpoint(phase_dir):
    ckpts = glob.glob(f"{phase_dir}/**/*.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint trovato in {phase_dir}")
    epoch_ckpts = [c for c in ckpts if "last.ckpt" not in os.path.basename(c)]
    if not epoch_ckpts:
        return ckpts[0]
    def extract_epoch_and_step(filepath):
        epoch = int(m.group(1)) if (m := re.search(r'epoch=(\d+)', filepath)) else -1
        step  = int(m.group(1)) if (m := re.search(r'step=(\d+)',  filepath)) else -1
        return (epoch, step)
    return sorted(epoch_ckpts, key=extract_epoch_and_step)[-1]

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

CHUNK2_CKPT = find_best_epoch_checkpoint(phase_dir)
print(f"Riprendo da: {CHUNK2_CKPT}")

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {CHUNK2_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.0"
    f" --model.init_args.lr_mult 0.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 4"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}

PHASE1_CKPT = extract_weights(
    find_best_epoch_checkpoint(phase_dir),
    f"{SAVE_DIR}/eomt_phase1_head_only.bin"
)
print(f"Phase 1 done → {PHASE1_CKPT}")

Riprendo da: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only/epoch=2-step=2232-v1.ckpt
2026-05-26 16:56:43.500430: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase1_head_only exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0

#unfreezing

In [ ]:
import os, glob, re, torch
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase2_unfreeze_last"
os.makedirs(phase_dir, exist_ok=True)

PHASE1_CKPT = f"{SAVE_DIR}/eomt_phase1_head_only.bin"
print(f"Esiste su Drive: {os.path.exists(PHASE1_CKPT)}")

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {PHASE1_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.3"
    f" --model.init_args.lr_mult 1.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 3"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}


Esiste su Drive: True
2026-05-27 15:20:21.329169: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

   | Name                     | Type                        | Params | Mode 
-----------------------------------------------------------------------

In [ ]:
import os, glob, re, torch
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase2_unfreeze_last"

def find_best_epoch_checkpoint(d):
    ckpts = glob.glob(f"{d}/**/*.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint trovato in {d}")
    epoch_ckpts = [c for c in ckpts if "last.ckpt" not in os.path.basename(c)]
    if not epoch_ckpts:
        return ckpts[0]
    def key(fp):
        e = int(m.group(1)) if (m := re.search(r'epoch=(\d+)', fp)) else -1
        s = int(m.group(1)) if (m := re.search(r'step=(\d+)',  fp)) else -1
        return (e, s)
    return sorted(epoch_ckpts, key=key)[-1]

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

CHUNK_A_CKPT = find_best_epoch_checkpoint(phase_dir)
print(f"Riprendo da: {CHUNK_A_CKPT}")

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {CHUNK_A_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.3"
    f" --model.init_args.lr_mult 1.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 2"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}

PHASE2_CKPT = extract_weights(
    find_best_epoch_checkpoint(phase_dir),
    f"{SAVE_DIR}/eomt_phase2_unfreeze_last.bin"
)
print(f"Phase 2 done → {PHASE2_CKPT}")


Riprendo da: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase2_unfreeze_last/epoch=2-step=2232.ckpt
2026-05-27 17:45:24.321673: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase2_unfreeze_last exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICE

In [ ]:
import os, glob, re, torch
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase3_unfreeze_more"
os.makedirs(phase_dir, exist_ok=True)

PHASE2_CKPT = f"{SAVE_DIR}/eomt_phase2_unfreeze_last.bin"
print(f"Esiste su Drive: {os.path.exists(PHASE2_CKPT)}")

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {PHASE2_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.6"
    f" --model.init_args.lr_mult 1.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 2"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}



Esiste su Drive: True
2026-05-28 14:42:27.293761: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

   | Name                     | Type                        | Params | Mode 
-----------------------------------------------------------------------

In [ ]:
import os, glob, re, torch
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase3_unfreeze_more"

def find_best_epoch_checkpoint(d):
    ckpts = glob.glob(f"{d}/**/*.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint trovato in {d}")
    epoch_ckpts = [c for c in ckpts if "last.ckpt" not in os.path.basename(c)]
    if not epoch_ckpts:
        return ckpts[0]
    def key(fp):
        e = int(m.group(1)) if (m := re.search(r'epoch=(\d+)', fp)) else -1
        s = int(m.group(1)) if (m := re.search(r'step=(\d+)',  fp)) else -1
        return (e, s)
    return sorted(epoch_ckpts, key=key)[-1]

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

CHUNK_A_CKPT = find_best_epoch_checkpoint(phase_dir)
print(f"Riprendo da: {CHUNK_A_CKPT}")

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {CHUNK_A_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.6"
    f" --model.init_args.lr_mult 1.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 3"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}

PHASE3_CKPT = extract_weights(
    find_best_epoch_checkpoint(phase_dir),
    f"{SAVE_DIR}/eomt_phase3_unfreeze_more.bin"
)
print(f"Phase 3 done → {PHASE3_CKPT}")


Riprendo da: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase3_unfreeze_more/epoch=1-step=1488.ckpt
2026-05-28 17:46:06.188175: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase3_unfreeze_more exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICE

AttributeError: module 'torch' has no attribute '_utils'

In [ ]:
import os, glob, re, torch
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase4_full"
os.makedirs(phase_dir, exist_ok=True)

PHASE3_CKPT = f"{SAVE_DIR}/eomt_phase3_unfreeze_more.bin"
print(f"Esiste su Drive: {os.path.exists(PHASE3_CKPT)}")

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {PHASE3_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.8"
    f" --model.init_args.lr_mult 1.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 2"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}


Esiste su Drive: True
2026-05-29 14:06:57.882013: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

   | Name                     | Type                        | Params | Mode 
-----------------------------------------------------------------------

In [ ]:
import os, glob, re, torch
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE      = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared"
DATA_PATH = BASE + "/datasets/cityscapes"
SAVE_DIR  = BASE + "/checkpoints/finetuned_v2"
CONFIG    = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
phase_dir = f"{SAVE_DIR}/phase4_full"

def find_best_epoch_checkpoint(d):
    ckpts = glob.glob(f"{d}/**/*.ckpt", recursive=True)
    if not ckpts:
        raise FileNotFoundError(f"Nessun checkpoint trovato in {d}")
    epoch_ckpts = [c for c in ckpts if "last.ckpt" not in os.path.basename(c)]
    if not epoch_ckpts:
        return ckpts[0]
    def key(fp):
        e = int(m.group(1)) if (m := re.search(r'epoch=(\d+)', fp)) else -1
        s = int(m.group(1)) if (m := re.search(r'step=(\d+)',  fp)) else -1
        return (e, s)
    return sorted(epoch_ckpts, key=key)[-1]

def extract_weights(ckpt_path, out_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save(state_dict, out_path)
    print(f"Weights saved → {out_path}")
    return out_path

CHUNK_A_CKPT = find_best_epoch_checkpoint(phase_dir)
print(f"Riprendo da: {CHUNK_A_CKPT}")

callback = (
    '{"class_path": "lightning.pytorch.callbacks.ModelCheckpoint", '
    '"init_args": {'
    f'"dirpath": "{phase_dir}", '
    '"save_last": true, '
    '"every_n_epochs": 1}}'
)

cmd = (
    f"python main.py fit"
    f" --config {CONFIG}"
    f" --model.init_args.ckpt_path {CHUNK_A_CKPT}"
    f" --model.init_args.load_ckpt_class_head True"
    f" --model.init_args.llrd 0.8"
    f" --model.init_args.lr_mult 1.0"
    f" --model.init_args.attn_mask_annealing_enabled False"
    f" --data.init_args.path {DATA_PATH}"
    f" --data.init_args.batch_size 2"
    f" --data.init_args.num_workers 2"
    f" --trainer.accumulate_grad_batches 2"
    f" --trainer.max_epochs 3"
    f' "--trainer.callbacks+={callback}"'
    f" --compile_disabled"
)
!{cmd}

PHASE4_CKPT = extract_weights(
    find_best_epoch_checkpoint(phase_dir),
    f"{SAVE_DIR}/eomt_phase4_full.bin"
)
print(f"Phase 4 done → {PHASE4_CKPT}")


Riprendo da: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase4_full/epoch=1-step=1488.ckpt
2026-05-29 15:46:35.351222: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 197 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/finetuned_v2/phase4_full exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `tr